# Motif-GNN Experiment Summary

This notebook tests a motif-augmented GNN for AML edge classification on the HI-Small dataset using a temporal split and class-imbalance weighting. It combines transaction attributes with streaming motif-style graph features and trains an edge classifier to detect laundering transactions. The key outputs are validation/test minority-class metrics and saved artifact metrics for comparison with other baselines.

In [36]:
import platform
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# -------- runtime + GPU + data setup --------
IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

GPU_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else None

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("Running in Colab:", IS_COLAB)
print("CUDA available:", GPU_AVAILABLE)
if GPU_AVAILABLE:
    print("GPU:", GPU_NAME)

try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    if out.returncode == 0 and out.stdout.strip():
        print("nvidia-smi:")
        print(out.stdout.strip())
except Exception:
    pass

PROJECT_ROOT = Path("/content/drive/MyDrive/math-168") if IS_COLAB else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
TRANS_CSV = DATA_DIR / "HI-Small_Trans.csv"
ACCOUNTS_CSV = DATA_DIR / "HI-Small_accounts.csv"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Missing data directory: {DATA_DIR}")
if not TRANS_CSV.exists():
    raise FileNotFoundError(f"Missing file: {TRANS_CSV}")
if not ACCOUNTS_CSV.exists():
    raise FileNotFoundError(f"Missing file: {ACCOUNTS_CSV}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("Found required files:", TRANS_CSV.name, "and", ACCOUNTS_CSV.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.12
Torch: 2.10.0+cu128
Running in Colab: True
CUDA available: True
GPU: Tesla T4
nvidia-smi:
Tesla T4, 15360 MiB
PROJECT_ROOT: /content/drive/MyDrive/math-168
DATA_DIR: /content/drive/MyDrive/math-168/data
Found required files: HI-Small_Trans.csv and HI-Small_accounts.csv


In [38]:
# Load + clean data and build a temporal subset with enough positives.
# Goal: avoid tiny-positive subsets (which gave near-zero F1).
trans = pd.read_csv(TRANS_CSV)
accounts = pd.read_csv(ACCOUNTS_CSV, usecols=["Bank ID", "Account Number"])

normalize_bank = lambda s: s.astype(str).str.strip().str.lstrip("0").replace("", "0")

src_bank = normalize_bank(trans["From Bank"])
src_acct = trans["Account"].astype(str).str.strip()
dst_bank = normalize_bank(trans["To Bank"])
dst_acct = trans["Account.1"].astype(str).str.strip()

df_all = pd.DataFrame(
    {
        "src": src_bank + "_" + src_acct,
        "dst": dst_bank + "_" + dst_acct,
        "timestamp": pd.to_datetime(trans["Timestamp"], errors="coerce"),
        "label": pd.to_numeric(trans["Is Laundering"], errors="coerce").fillna(0).astype(int).clip(0, 1),
        "amount_received": pd.to_numeric(trans["Amount Received"], errors="coerce").fillna(0.0),
        "amount_paid": pd.to_numeric(trans["Amount Paid"], errors="coerce").fillna(0.0),
        "receiving_currency_code": trans["Receiving Currency"].astype("category").cat.codes,
        "payment_currency_code": trans["Payment Currency"].astype("category").cat.codes,
        "payment_format_code": trans["Payment Format"].astype("category").cat.codes,
    }
).dropna(subset=["timestamp"])

acct_bank = normalize_bank(accounts["Bank ID"])
acct_num = accounts["Account Number"].astype(str).str.strip()
acct_keys = set((acct_bank + "_" + acct_num).tolist())
print("src coverage in accounts:", round(df_all["src"].isin(acct_keys).mean(), 4))
print("dst coverage in accounts:", round(df_all["dst"].isin(acct_keys).mean(), 4))
print("Full rows:", len(df_all), "positives:", int(df_all["label"].sum()))

# Temporal subset search: find smallest prefix that gives enough positives in
# overall data and each split (60/20/20).
MIN_TOTAL_POS = 350
MIN_SPLIT_POS = 40
START_EDGES = 300_000
STEP_EDGES = 100_000
MAX_EDGES = 1_200_000

df_all = df_all.sort_values("timestamp").reset_index(drop=True)
chosen_n = None
for n_try in range(START_EDGES, min(MAX_EDGES, len(df_all)) + 1, STEP_EDGES):
    sub = df_all.iloc[:n_try]
    y = sub["label"].to_numpy()
    total_pos = int(y.sum())
    tr, va = int(0.6 * n_try), int(0.8 * n_try)
    tr_pos = int(y[:tr].sum())
    va_pos = int(y[tr:va].sum())
    te_pos = int(y[va:].sum())
    if total_pos >= MIN_TOTAL_POS and min(tr_pos, va_pos, te_pos) >= MIN_SPLIT_POS:
        chosen_n = n_try
        break

if chosen_n is None:
    chosen_n = min(MAX_EDGES, len(df_all))

df = df_all.iloc[:chosen_n].copy().reset_index(drop=True)
y = df["label"].to_numpy()
tr, va = int(0.6 * len(df)), int(0.8 * len(df))
print(
    "Chosen rows:", len(df),
    "total_pos:", int(y.sum()),
    "train_pos:", int(y[:tr].sum()),
    "val_pos:", int(y[tr:va].sum()),
    "test_pos:", int(y[va:].sum()),
)

# Node indexing
node_ids = pd.Index(df["src"]).append(pd.Index(df["dst"])).unique().tolist()
node_to_idx = {n: i for i, n in enumerate(node_ids)}
df["src_idx"] = df["src"].map(node_to_idx).astype(np.int64)
df["dst_idx"] = df["dst"].map(node_to_idx).astype(np.int64)
df["ts"] = (df["timestamp"].astype("int64") / 1e9).astype(np.float64)

num_nodes = len(node_to_idx)
num_edges = len(df)
print("num_nodes:", num_nodes, "num_edges:", num_edges)

src coverage in accounts: 1.0
dst coverage in accounts: 1.0
Full rows: 5078345 positives: 5177
Chosen rows: 1200000 total_pos: 361 train_pos: 163 val_pos: 106 test_pos: 92
num_nodes: 444246 num_edges: 1200000


In [39]:
# Motif feature extraction + motif-aware edge GNN + training
# Paper-guided defaults used here:
# - temporal split: 60/20/20
# - hidden dim: 64, layers: 2 (Appendix E runtimes)
# - PNA-style LR/dropout range guidance (Table 11): lr in [1e-4,2e-3], dropout in [0,0.2]
# - minority class weighted loss (Table 11 tuned in [6,8])

import json
from collections import Counter, defaultdict

import torch.nn as nn
import torch.nn.functional as F

# -------- motif features (time-causal) --------
def compute_motif_features(src_idx, dst_idx, ts, window_hours=24.0, k=2):
    """
    Motif supports aligned to the 8 laundering patterns in the paper:
      0 fan_out        (a)
      1 fan_in         (b)
      2 gather_scatter (c)
      3 scatter_gather (d)
      4 cycle          (e)
      5 random         (f)
      6 bipartite      (g)
      7 stack          (h)

    Computed in streaming order with a rolling time window to avoid leakage.
    """
    window_seconds = window_hours * 3600.0
    n = len(src_idx)
    feats = np.zeros((n, 8), dtype=np.float32)
    order = np.argsort(ts, kind="stable")

    out_n = defaultdict(Counter)
    in_n = defaultdict(Counter)
    queue = []
    qh = 0

    def expire(cur_t):
        nonlocal qh
        threshold = cur_t - window_seconds
        while qh < len(queue) and queue[qh][0] < threshold:
            _, u_old, v_old = queue[qh]
            out_n[u_old][v_old] -= 1
            in_n[v_old][u_old] -= 1
            if out_n[u_old][v_old] <= 0:
                del out_n[u_old][v_old]
            if in_n[v_old][u_old] <= 0:
                del in_n[v_old][u_old]
            qh += 1

    for i in order:
        u = int(src_idx[i])
        v = int(dst_idx[i])
        t = float(ts[i])
        expire(t)

        out_u = out_n[u]
        in_u = in_n[u]
        out_v = out_n[v]
        in_v = in_n[v]

        # (a) Fan-out: source points to >=k distinct accounts.
        fan_out = max(0, len(out_u) - (k - 1))

        # (b) Fan-in: destination receives from >=k distinct accounts.
        fan_in = max(0, len(in_v) - (k - 1))

        # (c) Gather-scatter: same node exhibits both fan-in and fan-out.
        gather_scatter = min(max(0, len(in_u) - (k - 1)), max(0, len(out_u) - (k - 1)))

        # (d) Scatter-gather: shared intermediates w in u->w and w->v.
        scatter_gather = len(set(out_u).intersection(in_v))

        # (e) Cycle: direct reverse or short return support from v back to u.
        cycle = (1 if out_v.get(u, 0) > 0 else 0) + len(set(out_v).intersection(in_u))

        # (f) Random: continuation from v to other nodes (excluding immediate return to u).
        random_support = max(0, len(out_v) - (1 if u in out_v else 0))

        # (g) Bipartite proxy: many-to-many bridge around this edge.
        bipartite = max(0, len(in_u) - (k - 1)) * max(0, len(out_v) - (k - 1))

        # (h) Stack proxy: bipartite-like bridge plus one additional downstream layer.
        second_hop_nodes = set()
        for mid in out_v.keys():
            second_hop_nodes.update(out_n[mid].keys())
        stack = max(0, len(in_u) - (k - 1)) * max(0, len(second_hop_nodes) - (k - 1))

        feats[i] = np.array(
            [fan_out, fan_in, gather_scatter, scatter_gather, cycle, random_support, bipartite, stack],
            dtype=np.float32,
        )

        out_n[u][v] += 1
        in_n[v][u] += 1
        queue.append((t, u, v))

    return feats

# -------- model --------
class MotifEdgeGNN(nn.Module):
    def __init__(self, num_nodes, edge_dim, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, hidden)
        self.msg_layers = nn.ModuleList([nn.Linear(hidden + edge_dim, hidden) for _ in range(layers)])
        self.self_layers = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(layers)])
        self.nei_layers = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(layers)])
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2 + edge_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def encode_nodes(self, src, dst, edge_attr):
        h = self.node_emb.weight
        for msg_fc, self_fc, nei_fc in zip(self.msg_layers, self.self_layers, self.nei_layers):
            m_in = torch.cat([h[src], edge_attr], dim=1)
            m = F.relu(msg_fc(m_in))
            agg = torch.zeros_like(h)
            agg.index_add_(0, dst, m)
            h = F.relu(self_fc(h) + nei_fc(agg))
            h = self.drop(h)
        return h

    def forward(self, src, dst, edge_attr):
        h = self.encode_nodes(src, dst, edge_attr)
        edge_h = torch.cat([h[src], h[dst], edge_attr], dim=1)
        return self.classifier(edge_h).squeeze(1)

# -------- tensors --------
src_idx = df["src_idx"].values
dst_idx = df["dst_idx"].values
ts = df["ts"].values
labels = df["label"].values.astype(np.float32)

base_edge = df[[
    "amount_received",
    "amount_paid",
    "receiving_currency_code",
    "payment_currency_code",
    "payment_format_code",
]].values.astype(np.float32)

# 1 day motif window aligns with paper's non-scatter-gather feature windows.
motif = compute_motif_features(src_idx, dst_idx, ts, window_hours=24.0)
edge_attr = np.concatenate([base_edge, motif], axis=1)
edge_attr = (edge_attr - edge_attr.mean(0, keepdims=True)) / (edge_attr.std(0, keepdims=True) + 1e-8)

order = np.argsort(ts, kind="stable")
n = len(order)
tr, va = int(0.6 * n), int(0.8 * n)
train_idx = order[:tr]
val_idx = order[tr:va]
test_idx = order[va:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

src_t = torch.tensor(src_idx, dtype=torch.long, device=device)
dst_t = torch.tensor(dst_idx, dtype=torch.long, device=device)
edge_attr_t = torch.tensor(edge_attr, dtype=torch.float32, device=device)
y_t = torch.tensor(labels, dtype=torch.float32, device=device)

# Paper-guided defaults
HIDDEN_DIM = 64
NUM_LAYERS = 2
DROPOUT = 0.2
LR = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 12
PATIENCE = 3

model = MotifEdgeGNN(
    num_nodes=num_nodes,
    edge_dim=edge_attr_t.shape[1],
    hidden=HIDDEN_DIM,
    layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Keep weighting in the paper's tuned range [6, 8] while still adapting to imbalance.
pos = float(labels[train_idx].sum())
neg = float(len(train_idx) - pos)
auto_ratio = neg / max(pos, 1.0)
pos_weight_value = float(np.clip(auto_ratio, 6.0, 8.0))
pos_weight = torch.tensor(pos_weight_value, dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Train positives={int(pos)} negatives={int(neg)} pos_weight={pos_weight_value:.2f}")

def f1_at_threshold(y_true, y_prob, t=0.5):
    pred = (y_prob >= t).astype(np.int32)
    tp = ((pred == 1) & (y_true == 1)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    fn = ((pred == 0) & (y_true == 1)).sum()
    p = tp / (tp + fp + 1e-9)
    r = tp / (tp + fn + 1e-9)
    return 2 * p * r / (p + r + 1e-9), p, r

best_val_f1 = -1.0
best_state = None
best_thr = 0.5
stale = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    opt.zero_grad()
    logits = model(src_t, dst_t, edge_attr_t)
    loss = criterion(logits[torch.tensor(train_idx, dtype=torch.long, device=device)], y_t[torch.tensor(train_idx, dtype=torch.long, device=device)])
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(src_t, dst_t, edge_attr_t)).detach().cpu().numpy()

    y_val = labels[val_idx]
    p_val = probs[val_idx]
    cand = np.arange(0.05, 0.96, 0.01)
    f1s = [f1_at_threshold(y_val, p_val, t)[0] for t in cand]
    k = int(np.argmax(f1s))
    val_f1 = float(f1s[k])
    thr = float(cand[k])

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_thr = thr
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1

    print(f"epoch={epoch} loss={float(loss.item()):.4f} val_f1={val_f1:.4f} thr={thr:.2f}")
    if stale >= PATIENCE:
        print("Early stopping triggered")
        break

model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    test_probs = torch.sigmoid(model(src_t, dst_t, edge_attr_t)).detach().cpu().numpy()[test_idx]

test_f1, test_p, test_r = f1_at_threshold(labels[test_idx], test_probs, best_thr)
metrics = {
    "device": str(device),
    "num_nodes": int(num_nodes),
    "num_edges": int(num_edges),
    "positive_rate": float(labels.mean()),
    "train_pos": int(labels[train_idx].sum()),
    "val_pos": int(labels[val_idx].sum()),
    "test_pos": int(labels[test_idx].sum()),
    "pos_weight": float(pos_weight_value),
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "lr": LR,
    "epochs_ran": int(epoch),
    "best_val_f1": float(best_val_f1),
    "best_threshold": float(best_thr),
    "test_f1": float(test_f1),
    "test_precision": float(test_p),
    "test_recall": float(test_r),
}
print(json.dumps(metrics, indent=2))

OUT_DIR = PROJECT_ROOT / "artifacts"
OUT_DIR.mkdir(exist_ok=True)
OUT_JSON = OUT_DIR / "motif_gnn_quick_metrics.json"
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print("Saved:", OUT_JSON)

Training device: cuda
Train positives=163 negatives=719837 pos_weight=8.00
epoch=1 loss=0.6996 val_f1=0.0051 thr=0.51
epoch=2 loss=0.6076 val_f1=0.0100 thr=0.50
epoch=3 loss=0.5298 val_f1=0.0018 thr=0.48
epoch=4 loss=0.4610 val_f1=0.0008 thr=0.30
epoch=5 loss=0.3951 val_f1=0.0008 thr=0.22
Early stopping triggered
{
  "device": "cuda",
  "num_nodes": 444246,
  "num_edges": 1200000,
  "positive_rate": 0.00030083334422670305,
  "train_pos": 163,
  "val_pos": 106,
  "test_pos": 92,
  "pos_weight": 8.0,
  "hidden_dim": 64,
  "num_layers": 2,
  "dropout": 0.2,
  "lr": 0.001,
  "epochs_ran": 5,
  "best_val_f1": 0.010025062266417941,
  "best_threshold": 0.5000000000000001,
  "test_f1": 0.0,
  "test_precision": 0.0,
  "test_recall": 0.0
}
Saved: /content/drive/MyDrive/math-168/artifacts/motif_gnn_quick_metrics.json
